# Developing an AI-Powered Financial Chatbot
**BCG GenAI Job Simulation (Forage), Task 2**
**Prepared by:** Gibson Nyendwa, Junior Data Scientist, GenAI Consulting Team
**For:** Aisha, Senior Data Scientist | **Client:** Global Finance Corp. (GFC)

---

## Objective
Turn the Task 1 analysis of Microsoft, Tesla and Apple (FY2023 to FY2025) into a chatbot that answers financial questions in plain English, meeting GFC's four requirements: **efficiency, accuracy, a user-friendly interface and scalability**.

## Approach
I built the chatbot in two versions:
1. **Version 1: predefined queries.** A minimal rule-based bot that matches exact questions with `if`/`elif` logic. It proves the concept and shows where the project started.
2. **Version 2: the GFC Financial Chatbot.** A rule-based natural language understanding pipeline that handles questions phrased in many ways, follow-up questions, comparisons and health summaries, and explains one-off items rather than reporting bare numbers.

The code lives in `chatbot/financial_chatbot.py`, with 27 automated tests in `tests/test_chatbot.py`.

## 1. Load the Task 1 dataset

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from chatbot.financial_chatbot import FinancialChatbot, simple_chatbot, METRICS, CONTEXT_NOTES

df = pd.read_csv('../data/financial_analysis.csv')
df[['Company', 'Fiscal Year', 'Total Revenue', 'Net Income', 'Revenue Growth (%)', 'Net Income Growth (%)']]

,Company,Fiscal Year,Total Revenue,Net Income,Revenue Growth (%),Net Income Growth (%)
0,Apple,2023,383285,96995,NaN,NaN
1,Apple,2024,391035,93736,2.02,-3.36
2,Apple,2025,416161,112010,6.43,19.50
3,Microsoft,2023,211915,72361,NaN,NaN
4,Microsoft,2024,245122,88136,15.67,21.80
5,Microsoft,2025,281724,101832,14.93,15.54
6,Tesla,2023,96773,14997,NaN,NaN
7,Tesla,2024,97690,7091,0.95,-52.72
8,Tesla,2025,94827,3794,-2.93,-46.50


## 2. Version 1: predefined queries
The simplest chatbot: each supported question is matched exactly and mapped to an answer calculated from the data. Anything else gets a polite refusal.

In [2]:
predefined = [
    "What is the total revenue?",
    "How has net income changed over the last year?",
    "What are the total assets?",
    "What are the total liabilities?",
    "What is the cash flow from operating activities?",
    "What was Tesla's revenue?",   # not an exact predefined question
]
for q in predefined:
    print(f"You: {q}\nBot: {simple_chatbot(q)}\n")

You: What is the total revenue?
Bot: Microsoft's total revenue in FY2025 was $281.7bn.

You: How has net income changed over the last year?
Bot: Microsoft's net income rose 15.5% in FY2025.

You: What are the total assets?
Bot: Microsoft's total assets were $619.0bn at the end of FY2025.

You: What are the total liabilities?
Bot: Microsoft's total liabilities were $275.5bn at the end of FY2025.

You: What is the cash flow from operating activities?
Bot: Microsoft's cash flow from operating activities in FY2025 was $136.2bn.

You: What was Tesla's revenue?
Bot: Sorry, I can only provide information on predefined queries.



**What version 1 shows.** It works, but only for exact wording, one company and the latest year. The last question, which any client might ask, fails. For GFC's diverse client base, that would not be user-friendly enough, so I built version 2.

## 3. Version 2: how the GFC Financial Chatbot works
Every message goes through a five-step pipeline:

| Step | What happens | Example: *"How has Tesla's profit changed over the last year?"* |
|---|---|---|
| 1. Normalise | Lower case, remove punctuation | `how has tesla's profit changed over the last year` |
| 2. Extract entities | Find company, year and metric using keyword dictionaries | Company = Tesla, Year = FY2025 (latest), Metric = net income ("profit") |
| 3. Classify intent | Value, change, trend, comparison, summary or help | Intent = change ("changed") |
| 4. Use memory | Fill gaps from the previous turn for follow-ups | Not needed here |
| 5. Answer | Look up the verified data, write plain English, add context | Net income fell 46.5%, plus the reasons Tesla gave |

**Design choices linked to GFC's requirements:**
- **Accuracy:** every figure comes from the verified 10-K dataset. The bot declines questions about companies or years it has no data for, rather than guessing.
- **User-friendly:** synonyms are understood ("profit", "earnings" and "bottom line" all mean net income), figures appear in billions, and ratios come with a one-line explanation.
- **Context:** one-off items such as Apple's FY2024 tax charge are explained automatically, so users are not misled by a bare percentage.
- **Scalability:** the bot reads companies and years from the data file, so adding a company or a year means adding rows, not changing code.

### How the build applies the three principles of chatbot development

| Principle | What it means | Where it is in the code |
|---|---|---|
| **Rule-based logic** | If the user asks X, respond with Y | Version 1 maps exact questions with `if`/`elif`. Version 2 uses rules too, but on *meaning* rather than exact wording: keyword dictionaries identify the company, year, metric and intent, and each intent has its own response rule (`_answer_value`, `_answer_change`, `_answer_trend`, `_answer_compare`, `_answer_summary`). |
| **State management** | Remember the conversation's context | The `Memory` object stores the last company, metric, year and intent. "What about Microsoft?" or "And in 2024?" borrow whatever they leave out. A complete new question replaces the memory, so old context never leaks into an unrelated answer. |
| **Error handling** | Handle unrecognised queries gracefully and guide the user | Unrecognised questions get "Sorry, I didn't understand that" with a pointer to `help`. Companies or years outside the data are declined by name. A change for the first year explains why it can't be calculated and suggests a year that works. Empty messages are caught. |

In [3]:
print("Metrics the bot understands, and some of the words people use for them:\n")
for key, m in METRICS.items():
    print(f"  {m['label']:<30} {', '.join(m['keywords'][:5])}")

Metrics the bot understands, and some of the words people use for them:

  total revenue                  total revenue, revenue, revenues, sales, turnover
  net income                     net income, net profit, profit, profits, earnings
  total assets                   total assets, assets
  total liabilities              total liabilities, liabilities, debt, debts, obligations
  operating cash flow            cash flow from operating activities, operating cash flow, cash from operations, cash flow, operating cash
  net profit margin              net profit margin, profit margin, net margin, margin, margins
  liabilities-to-assets ratio    liabilities to assets, debt ratio, leverage, leveraged, gearing
  shareholders' equity           shareholders equity, shareholders' equity, equity, net worth, book value


## 4. Version 2 in action: a sample conversation
The same conversation a GFC client might have. Note the follow-up questions, which rely on the bot remembering the topic.

In [4]:
bot = FinancialChatbot()

conversation = [
    "Hello",
    "What was Apple's revenue in 2025?",
    "How has Tesla's net income changed over the last year?",
    "What about Microsoft?",
    "And in 2024?",
    "Which company has the highest net margin?",
    "Which company is least leveraged?",
    "Show Microsoft's operating cash flow trend",
    "How healthy is Tesla?",
    "Apple net income 2024",
]
for q in conversation:
    print(f"You: {q}\nBot: {bot.respond(q)}\n")

You: Hello
Bot: Hello! I'm the GFC Financial Chatbot. Ask me about Microsoft, Tesla or Apple, or type 'help'.

You: What was Apple's revenue in 2025?
Bot: Apple's total revenue in FY2025 was $416.2bn, which rose 6.4% on FY2024.

You: How has Tesla's net income changed over the last year?
Bot: Tesla's net income fell 46.5% in FY2025, from $7.1bn to $3.8bn.
  Context: Tesla linked the fall to lower vehicle deliveries and regulatory credit revenue, tariffs, and higher spending on AI and R&D.

You: What about Microsoft?
Bot: Microsoft's net income rose 15.5% in FY2025, from $88.1bn to $101.8bn.

You: And in 2024?
Bot: Microsoft's net income rose 21.8% in FY2024, from $72.4bn to $88.1bn.

You: Which company has the highest net margin?
Bot: Net profit margin in FY2025 (highest first):
  1. Microsoft: 36.1%
  2. Apple: 26.9%
  3. Tesla: 4.0%
Microsoft has the highest net profit margin. (Net profit margin measures the share of each dollar of revenue kept as profit.)

You: Which company is leas

## 5. Handling questions it cannot answer
For a financial tool, saying "I don't know" is far better than inventing a figure.

In [5]:
bot = FinancialChatbot()
for q in ["What is Amazon's revenue?", "What was Microsoft's revenue in 2019?",
          "What is the weather today?", "How did Apple's revenue change in 2023?"]:
    print(f"You: {q}\nBot: {bot.respond(q)}\n")

You: What is Amazon's revenue?
Bot: I don't have data for Amazon. I currently cover Microsoft, Tesla and Apple for FY2023 to FY2025.

You: What was Microsoft's revenue in 2019?
Bot: My data covers FY2023 to FY2025, so I can't answer for 2019. I won't guess figures I haven't verified.

You: What is the weather today?
Bot: Sorry, I didn't understand that. I answer questions about the financial performance of Microsoft, Tesla and Apple. Type 'help' for examples.

You: How did Apple's revenue change in 2023?
Bot: FY2023 is the first year in my data, so I can't calculate a change for it. Try asking about FY2024 or FY2025.



## 6. Help and guidance for new users

In [6]:
print(FinancialChatbot().respond("help"))

I can answer questions about Microsoft, Tesla and Apple, using their 10-K filings for FY2023 to FY2025.
  Metrics: revenue, net income, total assets, total liabilities, operating cash flow, net margin, leverage and shareholders' equity.
  Try asking:
  - What was Apple's revenue in 2025?
  - How has Tesla's net income changed over the last year?
  - Show Microsoft's operating cash flow trend
  - Which company has the highest net margin?
  - How healthy is Tesla?
  - What about Microsoft?  (follow-ups remember what we were discussing)
  Note: I report figures from company filings. This is not investment advice.


## 7. Testing
The test suite checks that answers match the dataset, follow-ups work, one-off items are explained and out-of-scope questions are declined.

In [7]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "tests"], capture_output=True, text=True, cwd="..")
print(result.stdout[-300:])

...........................                                              [100%]
27 passed in 0.30s



## 8. Limitations
Being clear about limits is part of delivering a reliable tool:
1. **Keyword matching, not true language understanding.** Unusual phrasing, spelling mistakes or complex multi-part questions may not be understood. The bot then says so rather than guessing.
2. **Limited data.** Three companies, three fiscal years and eight metrics. It cannot answer questions about quarters, segments, share prices or forecasts.
3. **Context notes are hand-written.** They cover the main one-off items found in the MD&A, and would need maintaining as new filings arrive.
4. **No forecasting or advice.** The bot reports historical figures only. It is not investment advice.
5. **Text interface.** It runs in a notebook or terminal. A client-facing version would need a web or app front end.

## 9. Next steps for GFC
1. **Automate data ingestion** from the SEC EDGAR XBRL API, so new 10-K and 10-Q filings update the bot automatically.
2. **Add a large language model (LLM) layer** for flexible understanding of any phrasing, with answers still grounded in the verified dataset (retrieval-augmented generation), so the model cannot invent figures.
3. **Build a web interface** so GFC's clients can use it without any technical knowledge.
4. **Expand coverage** to more companies, quarterly data and segment detail.

## 10. Conclusion
Version 2 meets GFC's brief: it answers in seconds rather than hours (**efficiency**), uses only verified 10-K data and refuses to guess (**accuracy**), understands everyday phrasing and explains its answers (**user-friendly**), and grows by adding data rather than code (**scalability**).